# WorldSim DGX Spark QLoRA Smoke Run

Thin Jupyter wrapper around the shared `training.lib.qlora_smoke` API. The notebook is designed to fail early on the wrong kernel and make post-run JSON structure problems operationally obvious.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
repo_marker = cwd / "training/lib/qlora_smoke.py"
assert repo_marker.exists(), f"Run this notebook from the repo root. cwd={cwd}"
{
    "python_executable": sys.executable,
    "cwd": str(cwd),
    "repo_marker": str(repo_marker),
}


## Environment Visibility

Run this cell first. If CUDA or bitsandbytes looks wrong here, stop before training.


In [ ]:
from training.lib.qlora_smoke import get_environment_summary

environment = get_environment_summary()
torch_info = environment.get("torch", {})
bnb_info = environment.get("bitsandbytes", {})
{
    "python_executable": environment.get("python", {}).get("executable"),
    "cwd": environment.get("cwd"),
    "torch_version": torch_info.get("version"),
    "torch_cuda_version": torch_info.get("cuda_version"),
    "torch_cuda_available": torch_info.get("cuda_available"),
    "gpu_count": torch_info.get("cuda_device_count", 0),
    "gpu_names": torch_info.get("cuda_device_names", []),
    "bitsandbytes_available": bnb_info.get("available"),
    "bitsandbytes_version": bnb_info.get("version"),
    "environment_summary": environment,
}


## Strict True-QLoRA Preflight

This uses the same shared runtime detection as the CLI path and fails immediately if true CUDA QLoRA cannot be satisfied.


In [ ]:
from training.lib.qlora_smoke import get_true_qlora_preflight

preflight = get_true_qlora_preflight()
assert preflight["ok"], preflight["blocker_reason"]
preflight


## Config

Edit smoke-run settings here only.


In [ ]:
from datetime import UTC, datetime
from pathlib import Path

from training.lib.qlora_smoke import SmokeRunConfig

run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
config = SmokeRunConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    train_file=Path("data/training/worldsim-v31-mix-v1/train_converted.jsonl"),
    dev_file=Path("data/training/worldsim-v31-mix-v1/dev_converted.jsonl"),
    output_dir=Path("outputs/smoke_cuda_notebook/worldsim-v31-mix-v1") / run_id,
    max_steps=5,
    max_train_samples=64,
    max_eval_samples=32,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    require_qlora=True,
    seed=42,
)
config.to_dict()


## Run Smoke Training


In [ ]:
from training.lib.qlora_smoke import run_smoke_or_raise

result = run_smoke_or_raise(config)
result.to_dict()


## Inspect Run Artifacts


In [ ]:
from training.lib.qlora_smoke import load_json_artifact, load_sample_generations, preview_metrics, summarize_sample_generations

run_config = load_json_artifact(result.output_dir, "run_config.json")
summary = load_json_artifact(result.output_dir, "summary.json")
metrics = preview_metrics(result.output_dir)
samples = load_sample_generations(result.output_dir)
sample_summary = summarize_sample_generations(samples)

{
    "run_config": run_config,
    "summary": summary,
    "metrics": metrics,
    "sample_summary": sample_summary,
}


In [ ]:
{
    "raw_parseable_json": sample_summary.get("raw_parseable_json"),
    "fenced_json": sample_summary.get("fenced_json"),
    "fence_stripped_parseable_json": sample_summary.get("fence_stripped_parseable_json"),
    "recoverable_fenced_json": sample_summary.get("recoverable_fenced_json"),
    "malformed_json": sample_summary.get("malformed_json"),
    "enum_drift_total": sample_summary.get("enum_drift_total"),
    "classifications": sample_summary.get("classifications"),
}


In [ ]:
sample_summary.get("recoverable_examples", [])


In [ ]:
samples[:3]


In [ ]:
if summary.get("used_true_qlora") and sample_summary.get("raw_parseable_json") == 0 and sample_summary.get("recoverable_fenced_json", 0) > 0 and sample_summary.get("malformed_json") == 0 and sample_summary.get("enum_drift_total") == 0:
    recommended_next_action = "Fix generation formatting or decoding to suppress markdown fences; do not redesign dataset or trainer yet."
    operational_issue = "markdown_fencing_only"
elif sample_summary.get("malformed_json", 0) > 0:
    recommended_next_action = "Investigate generation formatting and decoding deeper; malformed JSON remains after fence stripping."
    operational_issue = "malformed_json"
elif sample_summary.get("enum_drift_total", 0) > 0:
    recommended_next_action = "Investigate post-train structural quality and enum compliance before broader smoke conclusions."
    operational_issue = "enum_drift"
else:
    recommended_next_action = "Generation structure looks usable; move to the next smoke-run check or a slightly longer run."
    operational_issue = "structurally_usable"

{
    "true_qlora_passed": summary.get("used_true_qlora"),
    "raw_json_parse_failed": sample_summary.get("raw_parseable_json") == 0,
    "raw_parseable_json": sample_summary.get("raw_parseable_json"),
    "fence_stripped_parseable_json": sample_summary.get("fence_stripped_parseable_json"),
    "recoverable_fenced_json": sample_summary.get("recoverable_fenced_json"),
    "malformed_json": sample_summary.get("malformed_json"),
    "enum_drift_total": sample_summary.get("enum_drift_total"),
    "operational_issue": operational_issue,
    "recommended_next_action": recommended_next_action,
    "output_dir": result.output_dir,
}
